In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
# -*- coding: utf-8 -*-
from utils import *


# PART 1


###### 2. config & spark #########################################################
cfg = read_cfg()

data_dir = Path(cfg["outputs"]["data_dir"])
data_dir.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    filename=data_dir / cfg["outputs"]["log_file"],
    format="%(asctime)s %(levelname)s %(message)s"
)
log = logging.getLogger(__name__)

conf = SparkConf() \
    .setAppName(cfg["spark"]["appName"]) \
    .setMaster(cfg["spark"]["master"]) \
    .set("spark.executor.instances", cfg["spark"]["executor_instances"]) \
    .set("spark.executor.cores", cfg["spark"]["executor_cores"]) \
    .set("spark.driver.memory", cfg["spark"]["driver_memory"]) \
    .set("spark.executor.memory", cfg["spark"]["executor_memory"]) \
    .set("spark.executor.memoryOverhead", cfg["spark"]["executor_memoryOverhead"]) \
    .set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .set("spark.sql.parquet.int96RebaseMode", "CORRECTED") \
    # .set("spark.sql.shuffle.partitions", "400") \
    # .set("spark.memory.fraction", "0.5") \
    # .set("spark.memory.storageFraction", "0.2")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
log.info("SPARK STARTED")

###### 3. source tables & helpers #################################################
train_start, train_end = cfg["dates"]["train_date_start"], cfg["dates"]["train_date_end"]
test_start,  test_end  = cfg["dates"]["test_date_start"],  cfg["dates"]["test_date_end"]

###### 4. make train and test tables
train = filter_inn(spark, train_start, train_end).orderBy(sf.rand())
test = filter_inn(spark, test_start, test_end)

log.info("SAVING TRAIN")
(train
 .write.mode("overwrite")
 .parquet("{}.{}".format(cfg["outputs"]["save_schema"], "train_interactions")))
log.info("SAVED TRAIN")

log.info("SAVING TEST")
(test
 .write.mode("overwrite")
 .parquet("{}.{}".format(cfg["outputs"]["save_schema"], "test_interactions")))
log.info("SAVED TEST")

spark.stop()

In [ ]:
# -*- coding: utf-8 -*-
from utils import *

# PART ***. ADDING CONTEXT WORDS.


###### 2. config & spark #########################################################
cfg = read_cfg()

data_dir = Path(cfg["outputs"]["data_dir"])
data_dir.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    filename=data_dir / cfg["outputs"]["log_file"],
    format="%(asctime)s %(levelname)s %(message)s"
)
log = logging.getLogger(__name__)

conf = SparkConf() \
    .setAppName(cfg["spark"]["appName"]) \
    .setMaster(cfg["spark"]["master"]) \
    .set("spark.executor.instances", cfg["spark"]["executor_instances"]) \
    .set("spark.executor.cores", cfg["spark"]["executor_cores"]) \
    .set("spark.driver.memory", cfg["spark"]["driver_memory"]) \
    .set("spark.executor.memory", cfg["spark"]["executor_memory"]) \
    .set("spark.executor.memoryOverhead", cfg["spark"]["executor_memoryOverhead"]) \
    .set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .set("spark.sql.parquet.int96RebaseMode", "CORRECTED") \
    .set("spark.sql.shuffle.partitions", "400") \
    .set("spark.memory.fraction", "0.5") \
    .set("spark.memory.storageFraction", "0.2")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
log.info("SPARK STARTED")

###### 3. source tables & helpers #################################################
train_start, train_end = cfg["dates"]["train_date_start"], cfg["dates"]["train_date_end"]
test_start,  test_end  = cfg["dates"]["test_date_start"],  cfg["dates"]["test_date_end"]

context_words = spark.read.parquet(cfg["paths"]["context_words"])

context_words = context_words.filter((sf.col("short_dt") >= train_start) & (sf.col("short_dt") <= train_end))

# spark.stop()

In [ ]:
from pyspark.ml.clustering import LDA
from pyspark.ml.feature import CountVectorizer
from pyspark.ml.functions import vector_to_array

lda_cfg = cfg.get("lda", {})
num_topics = int(lda_cfg.get("n_topics", 50))
vocab_size = int(lda_cfg.get("vocab_size", 5000))
min_df = int(lda_cfg.get("min_df", 5))
max_iter = int(lda_cfg.get("max_iter", 20))

log.info("STARTING CONTEXT LDA WITH %s TOPICS", num_topics)

docs = (
    context_words
    .groupBy("inn_dt")
    .agg(sf.collect_list("word_v2").alias("words"))
    .filter(sf.size("words") > 0)
)

vectorizer = CountVectorizer(
    inputCol="words",
    outputCol="features",
    vocabSize=vocab_size,
    minDF=min_df,
)
cv_model = vectorizer.fit(docs)
vectorized = cv_model.transform(docs)

lda = LDA(k=num_topics, maxIter=max_iter, featuresCol="features", seed=42)
lda_model = lda.fit(vectorized)

topic_distributions = lda_model.transform(vectorized)
topic_distributions = topic_distributions.withColumn("topic_array", vector_to_array("topicDistribution"))

topic_cols = [f"dt_topic_weight_{i}" for i in range(num_topics)]
for idx, col_name in enumerate(topic_cols):
    topic_distributions = topic_distributions.withColumn(col_name, sf.col("topic_array")[idx])

dt_topic_weights = (
    topic_distributions
    .select("inn_dt", *topic_cols)
    .fillna(0.0, subset=topic_cols)
)

output_path = f"{cfg['outputs']['save_schema']}.dt_topic_weights"
log.info("SAVING TOPIC WEIGHTS TO %s", output_path)
dt_topic_weights.write.mode("overwrite").parquet(output_path)

cv_model_path = f"{cfg['outputs']['save_schema']}.context_cv_model"
lda_model_path = f"{cfg['outputs']['save_schema']}.context_lda_model"
cv_model.write().overwrite().save(cv_model_path)
lda_model.write().overwrite().save(lda_model_path)

spark.stop()


In [4]:
# -*- coding: utf-8 -*-
from utils import *


#  PART 2


###### 2. config & spark #########################################################
cfg = read_cfg()

data_dir = Path(cfg["outputs"]["data_dir"])
data_dir.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    filename=data_dir / cfg["outputs"]["log_file"],
    format="%(asctime)s %(levelname)s %(message)s"
)
log = logging.getLogger(__name__)

conf = SparkConf() \
    .setAppName(cfg["spark"]["appName"]) \
    .setMaster(cfg["spark"]["master"]) \
    .set("spark.executor.instances", cfg["spark"]["executor_instances"]) \
    .set("spark.executor.cores", cfg["spark"]["executor_cores"]) \
    .set("spark.driver.memory", cfg["spark"]["driver_memory"]) \
    .set("spark.executor.memory", cfg["spark"]["executor_memory"]) \
    .set("spark.executor.memoryOverhead", cfg["spark"]["executor_memoryOverhead"]) \
    .set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .set("spark.sql.parquet.int96RebaseMode", "CORRECTED") \
    .set("spark.sql.shuffle.partitions", "400") \
    .set("spark.memory.fraction", "0.5") \
    .set("spark.memory.storageFraction", "0.2")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
log.info("SPARK STARTED")


df = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.train_interactions")

kt_stats = (
    df.groupBy("inn_kt")
    .agg(
        F.avg("c_sum_fin").alias("kt_avg_sum"),
        F.stddev("c_sum_fin").alias("kt_stddev_sum"),
        F.min("c_sum_fin").alias("kt_min_sum"),
        F.max("c_sum_fin").alias("kt_max_sum"),
        F.expr("percentile_approx(c_sum_fin, 0.5)").alias("kt_median_sum"),
        F.countDistinct("inn_dt").alias("kt_buyers_count"),
        F.skewness("c_sum_fin").alias("kt_skewness_sum"),
    )
).fillna({
    "kt_stddev_sum": 200_000.0,
    "kt_skewness_sum": 2.5,
})

dt_stats = (
    df.groupBy("inn_dt")
    .agg(
        F.avg("c_sum_fin").alias("dt_avg_sum"),
        F.stddev("c_sum_fin").alias("dt_stddev_sum"),
        F.min("c_sum_fin").alias("dt_min_sum"),
        F.max("c_sum_fin").alias("dt_max_sum"),
        F.expr("percentile_approx(c_sum_fin, 0.5)").alias("dt_median_sum"),
        F.countDistinct("inn_kt").alias("dt_buyers_count"),
        F.skewness("c_sum_fin").alias("dt_skewness_sum"),
    )
).fillna({
    "dt_stddev_sum": 200_000.0,
    "dt_skewness_sum": 2.5
})


# log transforming large-scale features
kt_large_stats = [
    "kt_avg_sum", "kt_stddev_sum", "kt_min_sum", "kt_max_sum", "kt_median_sum",
    "kt_buyers_count"
]

dt_large_stats = [
    "dt_avg_sum", "dt_stddev_sum", "dt_min_sum", "dt_max_sum", "dt_median_sum",
    "dt_buyers_count"
]

for c in kt_large_stats:
    kt_pass = kt_pass.withColumn(c, F.log1p(F.col(c)))
    
for c in dt_large_stats:
    dt_pass = dt_pass.withColumn(c, F.log1p(F.col(c)))

kt_stats.write.mode("overwrite").parquet("{}.{}".format(cfg["outputs"]["save_schema"], "kt_stats"))
dt_stats.write.mode("overwrite").parquet("{}.{}".format(cfg["outputs"]["save_schema"], "dt_stats"))

spark.stop()

25/05/30 19:17:10 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [5]:
# -*- coding: utf-8 -*-
from utils import *
from pyspark.sql.functions import split, size, when, col
# PART 3
# creating freatures from okved
# joining with features count
###### 2. config & spark #########################################################
cfg = read_cfg()
data_dir = Path(cfg["outputs"]["data_dir"])
data_dir.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    filename=data_dir / cfg["outputs"]["log_file"],
    format="%(asctime)s %(levelname)s %(message)s"
)
log = logging.getLogger(__name__)
conf = SparkConf() \
    .setAppName(cfg["spark"]["appName"]) \
    .setMaster(cfg["spark"]["master"]) \
    .set("spark.executor.instances", cfg["spark"]["executor_instances"]) \
    .set("spark.executor.cores", cfg["spark"]["executor_cores"]) \
    .set("spark.driver.memory", cfg["spark"]["driver_memory"]) \
    .set("spark.executor.memory", cfg["spark"]["executor_memory"]) \
    .set("spark.executor.memoryOverhead", cfg["spark"]["executor_memoryOverhead"]) \
    .set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .set("spark.sql.parquet.int96RebaseMode", "CORRECTED") \
    .set("spark.sql.shuffle.partitions", "400") \
    .set("spark.memory.fraction", "0.5") \
    .set("spark.memory.storageFraction", "0.2")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
log.info("SPARK STARTED")
train = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.train_interactions")
kt_okved_parts = split(col("okved_cd_kt"), r"\.")
dt_okved_parts = split(col("okved_cd_dt"), r"\.")
# adding okved columns to train
train = train.withColumn("okved_cd_kt_lvl1", when(size(kt_okved_parts) >= 1, kt_okved_parts[0])) \
             .withColumn("okved_cd_kt_lvl2", when(size(kt_okved_parts) >= 2, kt_okved_parts[1])) \
             .withColumn("okved_cd_kt_lvl3", when(size(kt_okved_parts) >= 3, kt_okved_parts[2])) \
             .withColumn("okved_cd_kt_lvl4", when(size(kt_okved_parts) >= 4, kt_okved_parts[3]))
train = train.withColumn("okved_cd_dt_lvl1", when(size(dt_okved_parts) >= 1, dt_okved_parts[0])) \
             .withColumn("okved_cd_dt_lvl2", when(size(dt_okved_parts) >= 2, dt_okved_parts[1])) \
             .withColumn("okved_cd_dt_lvl3", when(size(dt_okved_parts) >= 3, dt_okved_parts[2])) \
             .withColumn("okved_cd_dt_lvl4", when(size(dt_okved_parts) >= 4, dt_okved_parts[3]))
train = train.fillna({
    "okved_cd_kt_lvl1": -1,
    "okved_cd_kt_lvl2": -1,
    "okved_cd_kt_lvl3": -1,
    "okved_cd_kt_lvl4": -1,
    "okved_cd_dt_lvl1": -1,
    "okved_cd_dt_lvl2": -1,
    "okved_cd_dt_lvl3": -1,
    "okved_cd_dt_lvl4": -1
})
kt_cat = ["okved_cd_kt", "okato_cd_kt", "bic_kt_34", "bic_kt_56", "bic_kt_79",
          "num_kt_13", "num_kt_45", "num_kt_68",
          "okved_cd_kt_lvl1", "okved_cd_kt_lvl2", "okved_cd_kt_lvl3", "okved_cd_kt_lvl4"
         ]
dt_cat = ["okved_cd_dt", "okato_cd_dt", "bic_dt_34", "bic_dt_56", "bic_dt_79",
          "num_dt_13", "num_dt_45", "num_dt_68",
          "okved_cd_dt_lvl1", "okved_cd_dt_lvl2", "okved_cd_dt_lvl3", "okved_cd_dt_lvl4"
         ]
kt_num = [
    "kt_avg_sum", "kt_stddev_sum", "kt_min_sum", "kt_max_sum", "kt_median_sum",
    "kt_buyers_count", "kt_skewness_sum"
]
dt_num = [
    "dt_avg_sum", "dt_stddev_sum", "dt_min_sum", "dt_max_sum", "dt_median_sum",
    "dt_buyers_count", "dt_skewness_sum"
]
lda_cfg = cfg.get("lda", {})
n_topics = int(lda_cfg.get("n_topics", 50))
topic_cols = [f"dt_topic_weight_{i}" for i in range(n_topics)]
kt_pass = (train.groupBy("inn_kt")
           .agg(*[F.first(c, True).alias(c) for c in (kt_cat)]))
dt_pass = (train.groupBy("inn_dt")
           .agg(*[F.first(c, True).alias(c) for c in (dt_cat)]))
kt_buyers = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.kt_stats").select("inn_kt", *kt_num)
dt_buyers = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.dt_stats").select("inn_dt", *dt_num)
kt_pass = kt_pass.join(kt_buyers, "inn_kt")
dt_pass = dt_pass.join(dt_buyers, "inn_dt")
dt_topics_path = f"{cfg['outputs']['save_schema']}.dt_topic_weights"
dt_topics = spark.read.parquet(dt_topics_path)
dt_pass = dt_pass.join(dt_topics, "inn_dt", "left")
dt_pass = dt_pass.fillna(0.0, subset=topic_cols)
kt_pass.write.mode("overwrite").parquet("arnsdpsbx_t_team_fin_adviser.kt_pass")
dt_pass.write.mode("overwrite").parquet("arnsdpsbx_t_team_fin_adviser.dt_pass")
spark.stop()


25/05/30 19:19:53 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [ ]:
# -*- coding: utf-8 -*-
# PART 4
# indexing all cat features including inn_kt and inn_kt
# making final passport for every inn
from utils import *
###### 2. config & spark #########################################################
cfg = read_cfg()
data_dir = Path(cfg["outputs"]["data_dir"])
data_dir.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    filename=data_dir / cfg["outputs"]["log_file"],
    format="%(asctime)s %(levelname)s %(message)s"
)
log = logging.getLogger(__name__)
conf = SparkConf() \
    .setAppName(cfg["spark"]["appName"]) \
    .setMaster(cfg["spark"]["master"]) \
    .set("spark.executor.instances", cfg["spark"]["executor_instances"]) \
    .set("spark.executor.cores", cfg["spark"]["executor_cores"]) \
    .set("spark.driver.memory", cfg["spark"]["driver_memory"]) \
    .set("spark.executor.memory", cfg["spark"]["executor_memory"]) \
    .set("spark.executor.memoryOverhead", cfg["spark"]["executor_memoryOverhead"]) \
    .set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .set("spark.sql.parquet.int96RebaseMode", "CORRECTED") \
    .set("spark.sql.shuffle.partitions", "400") \
    .set("spark.memory.fraction", "0.5") \
    .set("spark.memory.storageFraction", "0.2")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
kt_cat = ["inn_kt", "okved_cd_kt", "okato_cd_kt", "bic_kt_34", "bic_kt_56", "bic_kt_79",
          "num_kt_13", "num_kt_45", "num_kt_68",
          "okved_cd_kt_lvl1", "okved_cd_kt_lvl2", "okved_cd_kt_lvl3", "okved_cd_kt_lvl4"
         ]
dt_cat = ["inn_dt", "okved_cd_dt", "okato_cd_dt", "bic_dt_34", "bic_dt_56", "bic_dt_79",
          "num_dt_13", "num_dt_45", "num_dt_68",
          "okved_cd_dt_lvl1", "okved_cd_dt_lvl2", "okved_cd_dt_lvl3", "okved_cd_dt_lvl4"
         ]
kt_num = [
    "kt_avg_sum", "kt_stddev_sum", "kt_min_sum", "kt_max_sum", "kt_median_sum",
    "kt_buyers_count", "kt_skewness_sum"
]
dt_num = [
    "dt_avg_sum", "dt_stddev_sum", "dt_min_sum", "dt_max_sum", "dt_median_sum",
    "dt_buyers_count", "dt_skewness_sum"
]
lda_cfg = cfg.get("lda", {})
n_topics = int(lda_cfg.get("n_topics", 50))
topic_cols = [f"dt_topic_weight_{i}" for i in range(n_topics)]
dt_num = dt_num + topic_cols
kt_stats = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.kt_pass")
dt_stats = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.dt_pass")
indexer_kt = StringIndexer(inputCols=kt_cat,
                           outputCols=[f"{c}_index" for c in kt_cat],
                           handleInvalid="keep")
model_kt = indexer_kt.fit(kt_stats)
model_kt.write().save("arnsdpsbx_t_team_fin_adviser.indexer_kt")
kt_features_indexed = model_kt.transform(kt_stats)
cols = kt_features_indexed.columns
selected_kt_cols = ["inn_kt"] + [c for c in cols if c in kt_num or c in [f"{col}_index" for col in kt_cat]]
kt_features_indexed.select(*selected_kt_cols).coalesce(1).write.mode("overwrite").parquet("arnsdpsbx_t_team_fin_adviser.kt_pass_indexed")
indexer_dt = StringIndexer(inputCols=dt_cat,
                           outputCols=[f"{c}_index" for c in dt_cat],
                           handleInvalid="keep")
model_dt = indexer_dt.fit(dt_stats)
model_dt.write().save("arnsdpsbx_t_team_fin_adviser.indexer_dt")
dt_features_indexed = model_dt.transform(dt_stats)
cols = dt_features_indexed.columns
selected_dt_cols = ["inn_dt"] + [c for c in cols if c in dt_num or c in [f"{col}_index" for col in dt_cat]]
dt_features_indexed.select(*selected_dt_cols).coalesce(1).write.mode("overwrite").parquet("arnsdpsbx_t_team_fin_adviser.dt_pass_indexed")
spark.stop()


In [7]:
# -*- coding: utf-8 -*-


# PART 6
# saving embeddings

from utils import *

###### 2. config & spark #########################################################
cfg = read_cfg()

data_dir = Path(cfg["outputs"]["data_dir"])
data_dir.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    filename=data_dir / cfg["outputs"]["log_file"],
    format="%(asctime)s %(levelname)s %(message)s"
)
log = logging.getLogger(__name__)

conf = SparkConf() \
    .setAppName(cfg["spark"]["appName"]) \
    .setMaster(cfg["spark"]["master"]) \
    .set("spark.executor.instances", cfg["spark"]["executor_instances"]) \
    .set("spark.executor.cores", cfg["spark"]["executor_cores"]) \
    .set("spark.driver.memory", cfg["spark"]["driver_memory"]) \
    .set("spark.executor.memory", cfg["spark"]["executor_memory"]) \
    .set("spark.executor.memoryOverhead", cfg["spark"]["executor_memoryOverhead"]) \
    .set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .set("spark.sql.parquet.int96RebaseMode", "CORRECTED") \
    .set("spark.sql.shuffle.partitions", "400") \
    .set("spark.memory.fraction", "0.5") \
    .set("spark.memory.storageFraction", "0.2")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
log.info("SPARK STARTED")

emb_cols = [f"embed_{i}" for i in range(256)]
emb = (spark.read.parquet(cfg["paths"]["embeddings"])
    .withColumn("embedding", sf.array(*emb_cols))
    .select("inn", "embedding"))

kt_idx = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.kt_pass_indexed")
dt_idx = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.dt_pass_indexed")

kt_emb = (kt_idx.join(emb.withColumnRenamed("inn", "inn_kt"), "inn_kt")
                  .select("inn_kt", "inn_kt_index", "embedding"))
dt_emb = (dt_idx.join(emb.withColumnRenamed("inn", "inn_dt"), "inn_dt")
                  .select("inn_dt", "inn_dt_index", "embedding"))

kt_emb.coalesce(1).write.mode("overwrite").parquet("arnsdpsbx_t_team_fin_adviser.kt_embeddings_indexed")
dt_emb.coalesce(1).write.mode("overwrite").parquet("arnsdpsbx_t_team_fin_adviser.dt_embeddings_indexed")

spark.stop()

25/05/30 19:23:57 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
25/05/30 19:27:48 WARN DAGScheduler: Broadcasting large task binary with size 33.0 MiB
25/05/30 19:28:28 WARN DAGScheduler: Broadcasting large task binary with size 66.7 MiB
25/05/30 19:31:52 WARN DAGScheduler: Broadcasting large task binary with size 38.8 MiB
25/05/30 19:32:32 WARN DAGScheduler: Broadcasting large task binary with size 78.2 MiB


In [8]:
# -*- coding: utf-8 -*-


# PART 7
# TEST DICT CREATION


from utils import *

###### 2. config & spark #########################################################
cfg = read_cfg()

data_dir = Path(cfg["outputs"]["data_dir"])
data_dir.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    filename=data_dir / cfg["outputs"]["log_file"],
    format="%(asctime)s %(levelname)s %(message)s"
)
log = logging.getLogger(__name__)
log.info("WE ARE STARTED")

conf = SparkConf() \
    .setAppName(cfg["spark"]["appName"]) \
    .setMaster(cfg["spark"]["master"]) \
    .set("spark.executor.instances", cfg["spark"]["executor_instances"]) \
    .set("spark.executor.cores", cfg["spark"]["executor_cores"]) \
    .set("spark.driver.memory", cfg["spark"]["driver_memory"]) \
    .set("spark.executor.memory", cfg["spark"]["executor_memory"]) \
    .set("spark.executor.memoryOverhead", cfg["spark"]["executor_memoryOverhead"]) \
    .set("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED") \
    .set("spark.sql.parquet.int96RebaseMode", "CORRECTED") \
    .set("spark.sql.shuffle.partitions", "400") \
    .set("spark.memory.fraction", "0.5") \
    .set("spark.memory.storageFraction", "0.2")
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")
log.info("SPARK STARTED")


test_df = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.test_interactions")

kt_df = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.kt_pass_indexed")
dt_df = spark.read.parquet("arnsdpsbx_t_team_fin_adviser.dt_pass_indexed")

test_df = test_df.join(
    dt_df.select("inn_dt", "inn_dt_index"),
    on="inn_dt", how="inner"
).join(
    kt_df.select("inn_kt", "inn_kt_index"),
    on="inn_kt", how="inner"
)
test_dict_df = test_df.groupBy("inn_dt_index")\
    .agg(sf.collect_set("inn_kt_index").alias("kt_set"), sf.count("*").alias("cnt"))\
    .filter(sf.col("cnt") >= 15)\
    .select("inn_dt_index", "kt_set")

test_dict_df.coalesce(1).write.mode("overwrite").parquet("arnsdpsbx_t_team_fin_adviser.test_dict")

spark.stop()

25/05/30 19:36:09 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.
25/05/30 19:36:43 WARN DAGScheduler: Broadcasting large task binary with size 38.8 MiB
25/05/30 19:36:44 WARN DAGScheduler: Broadcasting large task binary with size 33.0 MiB
25/05/30 19:37:23 WARN DAGScheduler: Broadcasting large task binary with size 38.8 MiB
25/05/30 19:37:44 WARN DAGScheduler: Broadcasting large task binary with size 71.8 MiB
25/05/30 19:38:22 WARN DAGScheduler: Broadcasting large task binary with size 111.2 MiB


In [14]:
# AFTER MOVING FILES ABOVE TO A NEW DIR
!hdfs dfs - -mkdir hdfs://arnsdpsbx/user/22685380_omega-sbrf-ru/data/data_22
!hdfs dfs -cp viewfs://SDP-leverkin-ap-ca-sbrf-ru-SberSovetnik-471b9a/user/22685380_omega-sbrf-ru/deepfm_data/data_21/* hdfs://arnsdpsbx/user/22685380_omega-sbrf-ru/data/data_21

mkdir: `hdfs://arnsdpsbx/user/22685380_omega-sbrf-ru/data/data_21': File exists


In [15]:
!hdfs getconf -confKey hadoop.security.authentication

kerberos


In [10]:
!hdfs dfs -cp viewfs://SDP-leverkin-ap-ca-sbrf-ru-SberSovetnik-471b9a/user/22685380_omega-sbrf-ru/arnsdpsbx_t_team_fin_adviser.unigram_weights_dict hdfs://arnsdpsbx/user/22685380_omega-sbrf-ru/data/data_21